# RAG System Assignment: Building a Retrieval-Augmented Generation Pipeline

In this assignment, you will build a complete RAG (Retrieval-Augmented Generation) system that combines semantic search with a language model to answer questions based on a knowledge base.

## Learning Objectives
- Implement semantic search using the same embedding model as the demo (all-MiniLM-L6-v2)
- Build a RAG pipeline that retrieves relevant context
- Generate answers using a language model of your choice (see model options below)
- Understand how retrieval and generation work together

## Technologies Used
- **Embedding Model**: sentence-transformers/all-MiniLM-L6-v2 (for semantic search)
- **Chat Model**: Your choice among gpt-oss:120b-cloud, gemma3:270m, gemma3:27B, qwen3:4b
- **Knowledge Base**: `knowledge_base.md` (provided text chunks)

## Part 1: Setup and Installation

First, we need to install the required libraries. Make sure you have:
- Python 3.8+
- Ollama installed and running (for local models)

**Important Notes:**
1. **Ollama Setup**: Before running this notebook, ensure Ollama is installed and pull the model you plan to use (see "Choose chat model" below). For example:
   ```bash
   ollama pull gemma3:270m
   # or: ollama pull gemma3:27b  ollama pull qwen3:4b
   ```
   For **gpt-oss:120b-cloud**, if it is served from a remote Ollama host, set the environment variable `OLLAMA_HOST` before starting (e.g. `export OLLAMA_HOST=https://your-cloud-endpoint`).

2. **Virtual Environment**: It's recommended to use a virtual environment to avoid dependency conflicts.
   ```bash
   python -m venv venv
   source venv/bin/activate  # On Windows: venv\Scripts\activate
   pip install -r requirements.txt
   ```

In [1]:
# Install required libraries (same as the demo notebook)
# Run this cell once
%pip install sentence-transformers numpy scikit-learn ollama

# Verify installation
try:
    from sentence_transformers import SentenceTransformer
    print("✓ sentence-transformers installed successfully")
except ImportError:
    print("✗ sentence-transformers installation failed. Try: pip install sentence-transformers")

try:
    import ollama
    print("✓ ollama installed successfully")
except ImportError:
    print("✗ ollama installation failed. Try: pip install ollama")

Note: you may need to restart the kernel to use updated packages.


/Users/carbonjo/Library/CloudStorage/Dropbox/BSC-teaching+/Semesters/Fall25/DSA587-DSAwithG-AI/Materials/SemanticSearchEngine/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ sentence-transformers installed successfully
✓ ollama installed successfully


## Part 2: Import Libraries

Import all necessary libraries for the RAG pipeline.

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import ollama
import re
from typing import List, Tuple

## Part 2b: Choose Chat Model

Set **CHAT_MODEL** to one of the supported models below. The RAG pipeline will use this model for answer generation.

- **gpt-oss:120b-cloud** — Cloud-hosted (ensure OLLAMA_HOST is set if using a remote Ollama server)
- **gemma3:270m** — Small, fast local model
- **gemma3:27B** — Larger local model (requires more RAM)
- **qwen3:4b** — 4B-parameter local model

In [ ]:
# Choose one of: 'gpt-oss:120b-cloud', 'gemma3:270m', 'gemma3:27B', 'qwen3:4b'
CHAT_MODEL = 'gemma3:270m'

print(f"Using chat model: {CHAT_MODEL}")

## Part 3: Load the Knowledge Base

Load and parse the knowledge base file. Each section (separated by `---`) represents a chunk of text.

In [3]:
def load_knowledge_base(file_path: str) -> List[str]:
    """
    Load and parse the knowledge base markdown file.
    Each section separated by '---' is treated as a chunk.
    
    Args:
        file_path: Path to the knowledge_base.md file
    
    Returns:
        List of text chunks
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Split by '---' separator and clean up
    chunks = content.split('---')
    
    # Remove empty chunks and strip whitespace
    chunks = [chunk.strip() for chunk in chunks if chunk.strip()]
    
    # Remove markdown headers from the first chunk
    if chunks:
        chunks[0] = re.sub(r'^#+.*\n', '', chunks[0], flags=re.MULTILINE).strip()
    
    return chunks

# Load the knowledge base
knowledge_chunks = load_knowledge_base('knowledge_base.md')
print(f"Loaded {len(knowledge_chunks)} chunks from knowledge base")
print(f"\nExample chunk (first 200 characters):")
print(knowledge_chunks[0][:200] + "...")

Loaded 23 chunks from knowledge base

Example chunk (first 200 characters):
This document contains information about artificial intelligence, machine learning, and related technologies. Each section represents a chunk of knowledge that can be used for semantic search and RAG ...


## Part 4: Initialize the Embedding Model

Load the same embedding model as the demo (all-MiniLM-L6-v2). This model will convert text chunks and queries into numerical vectors.

In [4]:
# Initialize the embedding model (same as the demo)
print("Loading embedding model (all-MiniLM-L6-v2)...")
print("Note: This may take a few minutes on first run as the model downloads.")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully!")

# Test the model
test_embedding = embedding_model.encode(['test sentence'], convert_to_tensor=True)
print(f"\nEmbedding dimension: {test_embedding.shape[1]}")

Loading embedding model (all-MiniLM-L6-v2)...
Note: This may take a few minutes on first run as the model downloads.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1948.24it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!

Embedding dimension: 384


## Part 5: Create Embeddings for Knowledge Base

Convert all knowledge base chunks into embeddings. These embeddings will be used for semantic search.

In [5]:
# Create embeddings for all knowledge base chunks
print("Creating embeddings for knowledge base chunks...")
print(f"Processing {len(knowledge_chunks)} chunks...")

# Encode all chunks (same API as the demo)
chunk_embeddings = embedding_model.encode(knowledge_chunks, convert_to_tensor=True).cpu().numpy()

print(f"\nCreated embeddings with shape: {chunk_embeddings.shape}")
print(f"Each chunk is represented by a {chunk_embeddings.shape[1]}-dimensional vector")

Creating embeddings for knowledge base chunks...
Processing 23 chunks...

Created embeddings with shape: (23, 384)
Each chunk is represented by a 384-dimensional vector


## Part 6: Implement Semantic Search Function

Create a function that retrieves the most relevant chunks for a given query using cosine similarity.

In [6]:
def retrieve_relevant_chunks(query: str, chunk_embeddings: np.ndarray, 
                            knowledge_chunks: List[str], top_k: int = 3) -> List[Tuple[str, float]]:
    """
    Retrieve the top-k most relevant chunks for a given query.
    
    Args:
        query: User's question or query
        chunk_embeddings: Pre-computed embeddings for all knowledge chunks
        knowledge_chunks: List of all knowledge base chunks
        top_k: Number of top chunks to retrieve
    
    Returns:
        List of tuples (chunk_text, similarity_score) for top-k chunks
    """
    # Create embedding for the query
    query_embedding = embedding_model.encode([query], convert_to_tensor=True).cpu().numpy()
    
    # Calculate cosine similarity between query and all chunks
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    
    # Get top-k indices
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    # Return top-k chunks with their similarity scores
    results = [(knowledge_chunks[idx], similarities[idx]) for idx in top_indices]
    
    return results

# Test the retrieval function
test_query = "What is machine learning?"
print(f"Testing retrieval with query: '{test_query}'")
print("-" * 70)

retrieved = retrieve_relevant_chunks(test_query, chunk_embeddings, knowledge_chunks, top_k=2)

for i, (chunk, score) in enumerate(retrieved, 1):
    print(f"\n[Rank {i}] Similarity Score: {score:.4f}")
    print(f"Chunk preview: {chunk[:150]}...")

Testing retrieval with query: 'What is machine learning?'
----------------------------------------------------------------------

[Rank 1] Similarity Score: 0.7636
Chunk preview: ## Machine Learning Fundamentals

Machine Learning is a subset of AI that enables computers to learn and improve from experience without being explici...

[Rank 2] Similarity Score: 0.5879
Chunk preview: ## Supervised Learning

Supervised Learning involves training a model on labeled data, where the correct answers are known. The algorithm learns to ma...


## Part 7: Build the RAG Pipeline

Now combine retrieval with generation. The RAG pipeline:
1. Takes a user question
2. Retrieves relevant context from the knowledge base
3. Formats a prompt with the question and context
4. Generates an answer using the language model

In [7]:
def create_rag_prompt(user_question: str, context_chunks: List[Tuple[str, float]]) -> str:
    """
    Create a prompt for the language model that includes the question and retrieved context.
    
    Args:
        user_question: The user's question
        context_chunks: List of (chunk_text, similarity_score) tuples
    
    Returns:
        Formatted prompt string
    """
    # Combine context chunks
    context_text = "\n\n".join([chunk for chunk, _ in context_chunks])
    
    prompt = f"""You are a helpful AI assistant. Answer the following question using only the information provided in the context below. If the context doesn't contain enough information to answer the question, say so.

Context:
{context_text}

Question: {user_question}

Answer:"""
    
    return prompt

def rag_query(user_question: str, chunk_embeddings: np.ndarray, 
              knowledge_chunks: List[str], top_k: int = 3) -> dict:
    """
    Complete RAG pipeline: retrieve context and generate answer.
    
    Args:
        user_question: User's question
        chunk_embeddings: Pre-computed embeddings for knowledge chunks
        knowledge_chunks: List of all knowledge base chunks
        top_k: Number of chunks to retrieve
    
    Returns:
        Dictionary with 'answer', 'retrieved_chunks', and 'scores'
    """
    # Step 1: Retrieve relevant chunks
    retrieved_chunks = retrieve_relevant_chunks(
        user_question, chunk_embeddings, knowledge_chunks, top_k=top_k
    )
    
    # Step 2: Create prompt with context
    prompt = create_rag_prompt(user_question, retrieved_chunks)
    
    # Step 3: Generate answer using Ollama
    try:
        response = ollama.chat(
            model=CHAT_MODEL,
            messages=[{'role': 'user', 'content': prompt}]
        )
        answer = response['message']['content']
    except Exception as e:
        answer = f"Error generating answer: {str(e)}"
    
    return {
        'answer': answer,
        'retrieved_chunks': [chunk for chunk, _ in retrieved_chunks],
        'scores': [score for _, score in retrieved_chunks]
    }

# Test the RAG pipeline
print("Testing RAG pipeline...")
print("=" * 70)

test_question = "What is the difference between supervised and unsupervised learning?"
result = rag_query(test_question, chunk_embeddings, knowledge_chunks, top_k=3)

print(f"\nQuestion: {test_question}")
print("\n" + "=" * 70)
print("RETRIEVED CONTEXT:")
print("=" * 70)
for i, (chunk, score) in enumerate(zip(result['retrieved_chunks'], result['scores']), 1):
    print(f"\n[Chunk {i}] (Similarity: {score:.4f})")
    print(chunk[:200] + "...")

print("\n" + "=" * 70)
print("GENERATED ANSWER:")
print("=" * 70)
print(result['answer'])

Testing RAG pipeline...

Question: What is the difference between supervised and unsupervised learning?

RETRIEVED CONTEXT:

[Chunk 1] (Similarity: 0.7133)
## Unsupervised Learning

Unsupervised Learning works with unlabeled data to discover hidden patterns or structures. Unlike supervised learning, there are no correct answers provided during training. ...

[Chunk 2] (Similarity: 0.5278)
## Supervised Learning

Supervised Learning involves training a model on labeled data, where the correct answers are known. The algorithm learns to map inputs to outputs by finding patterns in the tra...

[Chunk 3] (Similarity: 0.4170)
## Machine Learning Fundamentals

Machine Learning is a subset of AI that enables computers to learn and improve from experience without being explicitly programmed. It involves algorithms that can id...

GENERATED ANSWER:
Supervised learning involves training a model on labeled data, where the correct answers are known.
Unsupervised learning involves training a model o

## Part 8: Interactive Q&A

Try asking your own questions! Modify the question below and run the cell.

In [ ]:
# Ask your own question
user_question = "What is RAG and how does it work?"

print("=" * 70)
print(f"QUESTION: {user_question}")
print("=" * 70)

result = rag_query(user_question, chunk_embeddings, knowledge_chunks, top_k=3)

print("\n" + "=" * 70)
print("RETRIEVED CONTEXT:")
print("=" * 70)
for i, (chunk, score) in enumerate(zip(result['retrieved_chunks'], result['scores']), 1):
    print(f"\n[Chunk {i}] (Similarity: {score:.4f})")
    print(chunk[:150] + "...")

print("\n" + "=" * 70)
print("GENERATED ANSWER:")
print("=" * 70)
print(result['answer'])

## Part 9: Assignment Tasks

Complete the following tasks:

### Task 1: Experiment with Different Queries
Try at least 5 different questions and observe:
- How well does semantic search retrieve relevant chunks?
- Are the similarity scores reasonable?
- Does the generated answer use the retrieved context?

### Task 2: Analyze Retrieval Quality
For each query, examine:
- Which chunks were retrieved?
- Do they actually contain relevant information?
- What similarity scores did they receive?

### Task 3: Improve the Prompt
Modify the `create_rag_prompt` function to:
- Include similarity scores in the prompt
- Add instructions for citing sources
- Experiment with different prompt formats

### Task 4: Experiment with top_k
Try different values of `top_k` (1, 2, 3, 5) and observe:
- How does the number of retrieved chunks affect the answer quality?
- Is there a sweet spot?

### Task 5: Error Handling
Add error handling for:
- Cases where Ollama is not running
- Cases where no relevant chunks are found
- Cases where the model fails to generate an answer

### Task 6: Evaluation (Optional)
Create a simple evaluation:
- Define 5 questions with expected answers
- Compare generated answers with expected answers
- Calculate a simple accuracy metric

## Part 10: Reflection Questions

Answer these questions in a markdown cell:

1. **Retrieval Quality**: How effective was semantic search at finding relevant chunks? Give examples.

2. **Answer Quality**: Did the language model generate accurate answers based on the retrieved context? Were there any hallucinations?

3. **RAG Benefits**: What are the advantages of using RAG compared to just using a language model without retrieval?

4. **Challenges**: What challenges did you encounter? How did you address them?

5. **Improvements**: What improvements would you make to this RAG system?

In [ ]:
# Write your reflection answers here
# (Convert this to a markdown cell or add a new markdown cell below)

reflection = """
## Reflection Answers

### 1. Retrieval Quality:
# Your answer here

### 2. Answer Quality:
# Your answer here

### 3. RAG Benefits:
# Your answer here

### 4. Challenges:
# Your answer here

### 5. Improvements:
# Your answer here
"""

print(reflection)